In [ ]:
!pip -q uninstall -y torch torchvision torchaudio ultralytics
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
!pip -q install -U ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 139.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 94.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 143.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 43.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 55.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 79.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 163.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 MB 114.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.6/216.6 MB 59.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.2/287.2 MB 76.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 76.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.7/124.7 MB 109.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Tách train/val

In [ ]:
import os, random, shutil
from pathlib import Path

root_path_str = "/content/drive/MyDrive/yolo_waste/waste350"
root = Path(root_path_str)
img_dir = root/"images"
lbl_dir = root/"labels"

train_img = root/"images_train"
val_img   = root/"images_val"
train_lbl = root/"labels_train"
val_lbl   = root/"labels_val"
for d in [train_img, val_img, train_lbl, val_lbl]:
    d.mkdir(parents=True, exist_ok=True)

imgs = sorted([p for p in img_dir.iterdir() if p.suffix.lower() in [".jpg",".jpeg",".png"]])
random.seed(42)
random.shuffle(imgs)
split = int(0.8*len(imgs))
train_set, val_set = imgs[:split], imgs[split:]

def move_pair(p, out_img, out_lbl):
    shutil.copy2(p, out_img/p.name)
    txt = lbl_dir/(p.stem + ".txt")
    if txt.exists():
        shutil.copy2(txt, out_lbl/txt.name)
    else:
        (out_lbl/(p.stem + ".txt")).write_text("")

for p in train_set: move_pair(p, train_img, train_lbl)
for p in val_set:   move_pair(p, val_img, val_lbl)

print("train:", len(train_set), "val:", len(val_set))

train: 280 val: 70


In [ ]:
%%bash
mkdir -p /content/waste350/images/train /content/waste350/images/val
mkdir -p /content/waste350/labels/train /content/waste350/labels/val

# chuyển ảnh
mv /content/drive/MyDrive/yolo_waste/waste350/images_train/* /content/waste350/images/train/
mv /content/drive/MyDrive/yolo_waste/waste350/images_val/*   /content/waste350/images/val/

# chuyển nhãn
mv /content/drive/MyDrive/yolo_waste/waste350/labels_train/* /content/waste350/labels/train/
mv /content/drive/MyDrive/yolo_waste/waste350/labels_val/*   /content/waste350/labels/val/

In [ ]:
%%bash
cat << 'EOF' > /content/drive/MyDrive/yolo_waste/waste350/waste350.yaml
path: /content/waste350
train: images/train
val: images/val
nc: 3
names:
  0: plastic
  1: glass
  2: paper
EOF

In [ ]:
%%bash
rm -f /content/drive/MyDrive/yolo_waste/waste350/*.cache
rm -f /content/drive/MyDrive/yolo_waste/waste350/images_train.cache /content/drive/MyDrive/yolo_waste/waste350/images_val.cache

TRAIN LẦN 1 VỚI 350 ẢNH ĐÃ VẼ TAY

In [ ]:
from ultralytics import YOLO

# Print the content of the YAML file for inspection
print("--- Content of waste350.yaml ---")
!cat /content/drive/MyDrive/yolo_waste/waste350/waste350.yaml
print("------------------------------")

model = YOLO("yolov8n.pt")
model.train(
    data="/content/drive/MyDrive/yolo_waste/waste350/waste350.yaml",
    epochs=60,
    imgsz=640,
    batch=16,
    name="waste_train_350_fixed2",
    project="/content/drive/MyDrive/yolo_waste/runs"
)

--- Content of waste350.yaml ---
path: /content/waste350
train: images/train
val: images/val
nc: 3
names:
  0: plastic
  1: glass
  2: paper
------------------------------
Ultralytics 8.3.251 🚀 Python-3.12.12 torch-2.9.1+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/yolo_waste/waste350/waste350.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yo

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7dc8bc1e7aa0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04

AUTO-LABEL CHO 650 ẢNH

In [ ]:
#Lọc 350 ảnh ra
from pathlib import Path

manual_root = Path("/content/waste350")
manual_train = list((manual_root/"images/train").glob("*"))
manual_val   = list((manual_root/"images/val").glob("*"))
manual_stems = set([p.stem for p in manual_train + manual_val if p.suffix.lower() in [".jpg",".jpeg",".png"]])

print("manual total:", len(manual_stems))


manual total: 350


In [ ]:
#Lọc riêng 650 ảnh
import shutil, glob, os
from pathlib import Path

src = Path("/content/drive/MyDrive/yolo_waste/waste1000" )
out650 = Path("/content/drive/MyDrive/yolo_waste/auto650/images")
out650.mkdir(parents=True, exist_ok=True)

img_files = []
for ext in ("*.jpg","*.jpeg","*.png"):
    img_files += glob.glob(str(src/ext))
img_files = [Path(p) for p in sorted(img_files)]

kept = 0
for img in img_files:
    if img.stem in manual_stems:
        continue
    shutil.copy2(img, out650/img.name)
    kept += 1

print("images in 1000:", len(img_files))
print("copied for auto-label (should ~650):", kept)


images in 1000: 1000
copied for auto-label (should ~650): 650


In [ ]:
#Tiến hành Auto-label cho 650 ảnh
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/yolo_waste/runs/waste_train_350_fixed2/weights/best.pt")

model.predict(
    source="/content/drive/MyDrive/yolo_waste/auto650/images",
    conf=0.20,
    iou=0.5,
    save_txt=True,
    save=False,
    project="/content/drive/MyDrive/yolo_waste/auto650",
    name="pred_auto650"
)

print("auto labels saved at: /content/drive/MyDrive/yolo_waste/auto650/pred_auto650/labels")


image 1/650 /content/drive/MyDrive/yolo_waste/auto650/images/img_00060.jpg: 640x640 1 plastic, 8.4ms
image 2/650 /content/drive/MyDrive/yolo_waste/auto650/images/img_00065.jpg: 480x640 1 plastic, 39.5ms
image 3/650 /content/drive/MyDrive/yolo_waste/auto650/images/img_00067.jpg: 480x640 1 plastic, 1 glass, 6.2ms
image 4/650 /content/drive/MyDrive/yolo_waste/auto650/images/img_00068.jpg: 640x640 1 plastic, 7.9ms
image 5/650 /content/drive/MyDrive/yolo_waste/auto650/images/img_00070.jpg: 640x640 1 plastic, 7.2ms
image 6/650 /content/drive/MyDrive/yolo_waste/auto650/images/img_00072.jpg: 640x640 1 plastic, 1 glass, 7.3ms
image 7/650 /content/drive/MyDrive/yolo_waste/auto650/images/img_00077.jpg: 640x640 1 paper, 17.0ms
image 8/650 /content/drive/MyDrive/yolo_waste/auto650/images/img_00079.jpg: 640x640 1 plastic, 7.3ms
image 9/650 /content/drive/MyDrive/yolo_waste/auto650/images/img_00087.jpg: 640x640 1 plastic, 1 paper, 7.1ms
image 10/650 /content/drive/MyDrive/yolo_waste/auto650/images/i

BUILD DATASEST 1000 ẢNH



In [ ]:
#Tập hợp lại ảnh
import shutil
from pathlib import Path

stage2 = Path("/content/drive/MyDrive/yolo_waste/waste_stage2")

# copy manual train
for img in (manual_root/"images/train").glob("*"):
    if img.suffix.lower() not in [".jpg",".jpeg",".png"]:
        continue
    shutil.copy2(img, stage2/"images/train"/img.name)
    shutil.copy2(manual_root/"labels/train"/(img.stem+".txt"), stage2/"labels/train"/(img.stem+".txt"))

# copy manual val
for img in (manual_root/"images/val").glob("*"):
    if img.suffix.lower() not in [".jpg",".jpeg",".png"]:
        continue
    shutil.copy2(img, stage2/"images/val"/img.name)
    shutil.copy2(manual_root/"labels/val"/(img.stem+".txt"), stage2/"labels/val"/(img.stem+".txt"))

print("manual copied.")


manual copied.


In [ ]:
#Add lại 650 ảnh và label
import glob, os
from pathlib import Path
import shutil

auto_imgs = Path("/content/drive/MyDrive/yolo_waste/auto650/images")
auto_lbls = Path("/content/drive/MyDrive/yolo_waste/auto650/pred_auto650/labels")

added = 0
missing = 0
for img in auto_imgs.glob("*"):
    if img.suffix.lower() not in [".jpg",".jpeg",".png"]:
        continue
    lb = auto_lbls/(img.stem+".txt")
    if not lb.exists():
        missing += 1
        continue
    shutil.copy2(img, stage2/"images/train"/img.name)
    shutil.copy2(lb, stage2/"labels/train"/lb.name)
    added += 1

print("auto added:", added)
print("auto missing labels:", missing)
print("train images:", len(list((stage2/'images/train').glob('*'))))
print("val images:", len(list((stage2/'images/val').glob('*'))))


auto added: 640
auto missing labels: 10
train images: 920
val images: 70


TẠO LABEL RỖNG CHO CÁC ẢNH BỊ LỖI MISSING LABELS

In [ ]:
from pathlib import Path

AUTO_IMG_DIR = Path("/content/drive/MyDrive/yolo_waste/auto650/images")
AUTO_LBL_DIR = Path("/content/drive/MyDrive/yolo_waste/auto650/pred_auto650/labels")

#Tiến hành tạo label rỗng cho ảnh không có file txt
created = 0
missing = []
for ext in ("*.jpg","*.jpeg","*.png","*.bmp","*.webp"):
    for im in AUTO_IMG_DIR.glob(ext):
        txt = AUTO_LBL_DIR/f"{im.stem}.txt"
        if not txt.exists():
            txt.touch()
            created += 1
            missing.append(im.name)

print("created empty labels:", created)
print("examples:", missing[:10])


created empty labels: 10
examples: ['img_00957.jpg', 'img_00979.jpg', 'img_00860.jpg', 'img_00895.jpg', 'img_00672.jpg', 'img_00869.jpg', 'img_00459.jpg', 'img_00488.jpg', 'img_00574.jpg', 'img_00615.jpg']


In [ ]:
import shutil, random
from pathlib import Path

SEED = 42

MANUAL_IMG = Path("/content/drive/MyDrive/yolo_waste/waste350/images")
MANUAL_LBL = Path("/content/drive/MyDrive/yolo_waste/waste350/labels")
AUTO_IMG   = Path("/content/drive/MyDrive/yolo_waste/auto650/images")
AUTO_LBL   = Path("/content/drive/MyDrive/yolo_waste/auto650/pred_auto650/labels")

OUT = Path("/content/drive/MyDrive/yolo_waste/waste_1k_split")
if OUT.exists():
    shutil.rmtree(OUT)

(train_img, val_img, train_lbl, val_lbl) = (
    OUT/"images/train", OUT/"images/val", OUT/"labels/train", OUT/"labels/val"
)
for p in [train_img, val_img, train_lbl, val_lbl]:
    p.mkdir(parents=True, exist_ok=True)

def list_pairs(img_dir, lbl_dir):
    imgs=[]
    for ext in ("*.jpg","*.jpeg","*.png","*.bmp","*.webp"):
        imgs += list(img_dir.glob(ext))
    imgs = sorted(imgs)
    pairs=[]
    for im in imgs:
        txt = lbl_dir/f"{im.stem}.txt"
        if txt.exists():
            pairs.append((im, txt))
    return pairs

pairs = list_pairs(MANUAL_IMG, MANUAL_LBL) + list_pairs(AUTO_IMG, AUTO_LBL)
print("total pairs:", len(pairs))

random.seed(SEED)
random.shuffle(pairs)

n_val = 80
val_pairs = pairs[:n_val]
train_pairs = pairs[n_val:]

def copy_pair(im, txt, dst_img, dst_lbl):
    shutil.copy2(im, dst_img/im.name)
    shutil.copy2(txt, dst_lbl/txt.name)

for im, txt in train_pairs:
    copy_pair(im, txt, train_img, train_lbl)

for im, txt in val_pairs:
    copy_pair(im, txt, val_img, val_lbl)

print("train images:", len(list(train_img.glob("*.*"))))
print("val images:", len(list(val_img.glob("*.*"))))
print("train labels:", len(list(train_lbl.glob("*.txt"))))
print("val labels:", len(list(val_lbl.glob("*.txt"))))


total pairs: 1000
train images: 920
val images: 80
train labels: 920
val labels: 80


In [ ]:
%%bash
cat << 'EOF' > /content/drive/MyDrive/yolo_waste/waste_1k_split/data.yaml
path: /content/drive/MyDrive/yolo_waste/waste_1k_split
train: images/train
val: images/val

nc: 3
names:
  0: plastic
  1: glass
  2: paper
EOF


In [ ]:
#Train lần 2
from ultralytics import YOLO

model2 = YOLO("/content/drive/MyDrive/yolo_waste/runs/waste_train_350_fixed2/weights/best.pt")
model2.train(
    data="/content/drive/MyDrive/yolo_waste/waste_1k_split/data.yaml",
    epochs=40,
    imgsz=640,
    batch=16,
    name="waste_train_1k",
    project="/content/drive/MyDrive/yolo_waste/runs"
)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.2 🚀 Python-3.12.12 torch-2.9.1+cu126 CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/yolo_waste/waste_1k_split/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4

KeyboardInterrupt: 

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/yolo_waste/runs/waste_train_1k/weights/last.pt")
model.train(resume=True)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.2 🚀 Python-3.12.12 torch-2.9.1+cu126 CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/yolo_waste/waste_1k_split/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x785bec647ef0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/yolo_waste/runs/waste_train_1k/weights/best.pt")
model.train(
    data="/content/drive/MyDrive/yolo_waste/waste_1k_split/data.yaml",
    epochs=40,
    imgsz=640,
    batch=8,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    translate=0.1, scale=0.5, fliplr=0.5,
    mosaic=1.0,
    project="/content/drive/MyDrive/yolo_waste/runs",
    name="waste_train_1k_aug"
)

New https://pypi.org/project/ultralytics/8.4.4 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.3 🚀 Python-3.12.12 torch-2.9.1+cu126 CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/yolo_waste/waste_1k_split/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/yolo_waste/runs/waste_train_1k/weights/

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/yolo_waste/runs/waste_train_1k_aug2/weights/last.pt")
model.train(resume=True)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.5 🚀 Python-3.12.12 torch-2.9.1+cu126 CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/yolo_waste/waste_1k_split/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x791331c00ce0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04

In [ ]:
BEST = "/content/drive/MyDrive/yolo_waste/runs/waste_train_1k_aug2/weights/best.pt"

In [ ]:
from ultralytics import YOLO
import pandas as pd

model = YOLO(BEST)

#Validate trên val set trong data.yaml của bạn
metrics = model.val(data="/content/drive/MyDrive/yolo_waste/waste_1k_split/data.yaml", imgsz=640, conf=0.25)

#Ultralytics trả metric theo object; in ra các cái quan trọng
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

P = float(metrics.box.mp)
R = float(metrics.box.mr)
F1 = 0 if (P + R) == 0 else (2*P*R/(P+R))
print("F1-score:", F1)


Ultralytics 8.4.5 🚀 Python-3.12.12 torch-2.9.1+cu126 CPU (Intel Xeon CPU @ 2.20GHz)
Model summary (fused): 73 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.8±0.3 ms, read: 79.9±164.5 MB/s, size: 2489.0 KB)
val: Scanning /content/drive/MyDrive/yolo_waste/waste_1k_split/labels/val.cache... 80 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 80/80 17.7Mit/s 0.0s
val: /content/drive/MyDrive/yolo_waste/waste_1k_split/images/val/img_00280.jpg: corrupt JPEG restored and saved
val: /content/drive/MyDrive/yolo_waste/waste_1k_split/images/val/img_00502.jpg: corrupt JPEG restored and saved
val: /content/drive/MyDrive/yolo_waste/waste_1k_split/images/val/img_00507.jpg: corrupt JPEG restored and saved
val: /content/drive/MyDrive/yolo_waste/waste_1k_split/images/val/img_00798.jpg: corrupt JPEG restored and saved
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 4.6s/it 22.9s
                   

In [ ]:
import pandas as pd

summary = pd.DataFrame([{
    "Precision": P,
    "Recall": R,
    "F1": F1,
    "mAP@50": float(metrics.box.map50),
    "mAP@50-95": float(metrics.box.map),
}])

summary = summary.round(2)
summary


,Precision,Recall,F1,mAP@50,mAP@50-95
0,0.86,0.94,0.9,0.94,0.85


In [ ]:
#Predict vài ảnh demo
from ultralytics import YOLO
from pathlib import Path

model = YOLO(BEST)

DEMO_DIR = "/content/drive/MyDrive/yolo_waste/waste_1k_split/images/val"
OUT_DIR = "/content/drive/MyDrive/yolo_waste/predict_demo_4k"

model.predict(
    source=DEMO_DIR,
    conf=0.35,
    iou=0.5,
    save=True,
    save_txt=False,
    project=OUT_DIR,
    name="conf035"
)

print("Saved predictions at:", f"{OUT_DIR}/conf035")



image 1/80 /content/drive/MyDrive/yolo_waste/waste_1k_split/images/val/img_00002.jpg: 640x640 1 paper, 232.3ms
image 2/80 /content/drive/MyDrive/yolo_waste/waste_1k_split/images/val/img_00017.jpg: 640x640 1 paper, 194.1ms
image 3/80 /content/drive/MyDrive/yolo_waste/waste_1k_split/images/val/img_00019.png: 640x640 1 paper, 187.5ms
image 4/80 /content/drive/MyDrive/yolo_waste/waste_1k_split/images/val/img_00034.jpg: 640x640 2 glasss, 209.4ms
image 5/80 /content/drive/MyDrive/yolo_waste/waste_1k_split/images/val/img_00039.jpg: 640x640 1 glass, 197.7ms
image 6/80 /content/drive/MyDrive/yolo_waste/waste_1k_split/images/val/img_00050.png: 640x640 1 paper, 197.0ms
image 7/80 /content/drive/MyDrive/yolo_waste/waste_1k_split/images/val/img_00054.jpg: 640x640 1 paper, 199.7ms
image 8/80 /content/drive/MyDrive/yolo_waste/waste_1k_split/images/val/img_00057.png: 640x640 4 plastics, 198.2ms
image 9/80 /content/drive/MyDrive/yolo_waste/waste_1k_split/images/val/img_00060.jpg: 640x640 1 plastic, 28